In [0]:
silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_truck_utilization = spark.table(
    "workspace.transportation_analytics.silver_truck_utilization_metrics"
)

silver_trucks.printSchema()
silver_truck_utilization.printSchema()

In [0]:
from pyspark.sql import functions as F

gold_fleet_utilization = (
    silver_truck_utilization
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "fuel_type",
            "status",
            "home_terminal"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_fleet_utilization)

In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .groupBy(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "fuel_type",
        "status",
        "home_terminal"
    )
    .agg(
        F.sum("trips_completed").alias("total_trips"),
        F.sum("total_miles").alias("total_miles"),
        F.sum("total_revenue").alias("total_revenue"),
        F.avg("utilization_rate").alias("avg_utilization_rate"),
        F.avg("average_mpg").alias("avg_mpg"),
        F.sum("maintenance_cost").alias("total_maintenance_cost"),
        F.sum("downtime_hours").alias("total_downtime_hours")
    )
)

display(gold_fleet_utilization)

In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
    )
)

display(
    gold_fleet_utilization.select(
        "truck_id",
        "unit_number",
        "total_trips",
        "total_miles",
        "total_revenue",
        "revenue_per_mile",
        "avg_utilization_rate",
        "avg_mpg",
        "total_downtime_hours"
    )
)

In [0]:
gold_fleet_utilization = (
    gold_fleet_utilization
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
    )
)

display(gold_fleet_utilization)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_fleet_utilization"
)

target.alias("t").merge(
    gold_fleet_utilization.alias("s"),
    "t.truck_id = s.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Fleet Utilization table updated using MERGE")

In [0]:
display(
    gold_fleet_utilization
    .orderBy(F.desc("total_revenue"))
    .select(
        "truck_id",
        "unit_number",
        "total_miles",
        "total_revenue",
        "revenue_per_mile"
    )
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.